# Notebook 4 – Adult BMI Distribution

## Objective
Download, clean, quality check, and export age-standardized adult BMI distribution data for Pacific Island jurisdictions.

### Source
NCD Risk Factor Collaboration (NCD-RisC)

### Study Population
16 Pacific Island jurisdictions

### Output
processed_data/adult_bmi_distribution.csv

## 1. Initialize project and download source data

In [1]:
import pandas as pd
import requests
from io import StringIO
import os

os.makedirs("raw_data/NCD_RisC", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)

url = "https://www.ncdrisc.org/downloads/bmi-2026/adult/NCD_RisC_Nature_2026_BMI_age_standardised_country.csv"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=60)

print("Status code:", response.status_code)
print("Downloaded MB:", round(len(response.content) / 1_000_000, 2))

response.raise_for_status()

bmi_raw = pd.read_csv(StringIO(response.text))

# Fix malformed Year column name caused by encoding artifact
bmi_raw = bmi_raw.rename(columns={bmi_raw.columns[0]: "Year"})

bmi_raw.to_csv(
    "raw_data/NCD_RisC/ncdrisc_bmi_distribution_raw.csv",
    index=False
)

print("Rows downloaded:", len(bmi_raw))
print("Columns:")
print(bmi_raw.columns.tolist())

bmi_raw.head()

Status code: 200
Downloaded MB: 9.23
Rows downloaded: 18000
Columns:
['Year', 'Sex', 'Country/Region/World', 'ISO', 'Prevalence of BMI<18.5 kg/mÂ² (underweight)', 'Prevalence of BMI<18.5 kg/mÂ² (underweight) lower 95% uncertainty interval', 'Prevalence of BMI<18.5 kg/mÂ² (underweight) upper 95% uncertainty interval', 'Prevalence of BMI>=30 kg/mÂ² (obesity)', 'Prevalence of BMI>=30 kg/mÂ² (obesity) lower 95% uncertainty interval', 'Prevalence of BMI>=30 kg/mÂ² (obesity) upper 95% uncertainty interval', 'Prevalence of BMI 18.5 kg/mÂ² to <20 kg/mÂ²', 'Prevalence of BMI 18.5 kg/mÂ² to <20 kg/mÂ² lower 95% uncertainty interval', 'Prevalence of BMI 18.5 kg/mÂ² to <20 kg/mÂ² upper 95% uncertainty interval', 'Prevalence of BMI 20 kg/mÂ² to <25 kg/mÂ²', 'Prevalence of BMI 20 kg/mÂ² to <25 kg/mÂ² lower 95% uncertainty interval', 'Prevalence of BMI 20 kg/mÂ² to <25 kg/mÂ² upper 95% uncertainty interval', 'Prevalence of BMI 25 kg/mÂ² to <30 kg/mÂ²', 'Prevalence of BMI 25 kg/mÂ² to <30 kg/mÂ² lower

,Year,Sex,Country/Region/World,ISO,Prevalence of BMI<18.5 kg/mÂ² (underweight),Prevalence of BMI<18.5 kg/mÂ² (underweight) lower 95% uncertainty interval,Prevalence of BMI<18.5 kg/mÂ² (underweight) upper 95% uncertainty interval,Prevalence of BMI>=30 kg/mÂ² (obesity),Prevalence of BMI>=30 kg/mÂ² (obesity) lower 95% uncertainty interval,Prevalence of BMI>=30 kg/mÂ² (obesity) upper 95% uncertainty interval,...,Prevalence of BMI 25 kg/mÂ² to <30 kg/mÂ² upper 95% uncertainty interval,Prevalence of BMI 30 kg/mÂ² to <35 kg/mÂ²,Prevalence of BMI 30 kg/mÂ² to <35 kg/mÂ² lower 95% uncertainty interval,Prevalence of BMI 30 kg/mÂ² to <35 kg/mÂ² upper 95% uncertainty interval,Prevalence of BMI 35 kg/mÂ² to <40 kg/mÂ²,Prevalence of BMI 35 kg/mÂ² to <40 kg/mÂ² lower 95% uncertainty interval,Prevalence of BMI 35 kg/mÂ² to <40 kg/mÂ² upper 95% uncertainty interval,Prevalence of BMI >=40 kg/mÂ² (morbid obesity),Prevalence of BMI >=40 kg/mÂ² (morbid obesity) lower 95% uncertainty interval,Prevalence of BMI >=40 kg/mÂ² (morbid obesity) upper 95% uncertainty interval
0,1980,Women,Afghanistan,AFG,0.349178,0.212458,0.490472,0.011151,0.003766,0.025658,...,0.081773,0.007842,0.001678,0.021491,0.002101,0.000298,0.006539,0.001207,0.000114,0.004371
1,1981,Women,Afghanistan,AFG,0.342155,0.210278,0.479112,0.011999,0.004274,0.026569,...,0.084384,0.008468,0.002001,0.022284,0.002260,0.000362,0.006743,0.001270,0.000140,0.004387
2,1982,Women,Afghanistan,AFG,0.334917,0.207405,0.467320,0.012924,0.004915,0.027618,...,0.088158,0.009150,0.002355,0.023096,0.002433,0.000431,0.006971,0.001340,0.000167,0.004437
3,1983,Women,Afghanistan,AFG,0.327480,0.205469,0.456376,0.013932,0.005586,0.028896,...,0.091604,0.009893,0.002756,0.024217,0.002622,0.000508,0.007240,0.001417,0.000197,0.004486
4,1984,Women,Afghanistan,AFG,0.319881,0.202980,0.444182,0.015030,0.006303,0.030322,...,0.095513,0.010701,0.003216,0.025197,0.002828,0.000597,0.007455,0.001501,0.000233,0.004532


## 2. Filter to Pacific Island jurisdictions

In [2]:
pacific_iso3 = [
    "ASM", "COK", "FJI", "FSM", "PYF",
    "KIR", "MHL", "NRU", "NIU",
    "PLW", "WSM", "SLB", "TKL",
    "TON", "TUV", "VUT"
]

bmi_pacific = bmi_raw[
    bmi_raw["ISO"].isin(pacific_iso3)
].copy()

bmi_pacific = bmi_pacific.rename(columns={
    "Country/Region/World": "Country",
    "ISO": "ISO3",
    "Prevalence of BMI<18.5 kg/mÂ² (underweight)": "Underweight_Pct",
    "Prevalence of BMI 20 kg/mÂ² to <25 kg/mÂ²": "HealthyWeight_Pct",
    "Prevalence of BMI 25 kg/mÂ² to <30 kg/mÂ²": "Overweight_Pct",
    "Prevalence of BMI>=30 kg/mÂ² (obesity)": "Obesity_Pct",
    "Prevalence of BMI 30 kg/mÂ² to <35 kg/mÂ²": "Class1_Obesity_Pct",
    "Prevalence of BMI 35 kg/mÂ² to <40 kg/mÂ²": "Class2_Obesity_Pct",
    "Prevalence of BMI >=40 kg/mÂ² (morbid obesity)": "Morbid_Obesity_Pct"
})

# Average men and women
bmi_pacific = (
    bmi_pacific
    .groupby(["ISO3", "Country", "Year"], as_index=False)
    .agg({
        "Underweight_Pct": "mean",
        "HealthyWeight_Pct": "mean",
        "Overweight_Pct": "mean",
        "Obesity_Pct": "mean",
        "Class1_Obesity_Pct": "mean",
        "Class2_Obesity_Pct": "mean",
        "Morbid_Obesity_Pct": "mean"
    })
)

print("Rows:", len(bmi_pacific))
bmi_pacific.head(20)

Rows: 720


,ISO3,Country,Year,Underweight_Pct,HealthyWeight_Pct,Overweight_Pct,Obesity_Pct,Class1_Obesity_Pct,Class2_Obesity_Pct,Morbid_Obesity_Pct
0,ASM,American Samoa,1980,0.007469,0.153164,0.283307,0.541396,0.289991,0.160192,0.091213
1,ASM,American Samoa,1981,0.007158,0.146564,0.279242,0.553142,0.292090,0.165350,0.095702
2,ASM,American Samoa,1982,0.006859,0.140056,0.274968,0.564962,0.293980,0.170602,0.100379
3,ASM,American Samoa,1983,0.006574,0.133713,0.270542,0.576719,0.295630,0.175892,0.105196
4,ASM,American Samoa,1984,0.006300,0.127600,0.265996,0.588321,0.297023,0.181171,0.110127
5,ASM,American Samoa,1985,0.006037,0.121765,0.261393,0.599653,0.298142,0.186363,0.115148
6,ASM,American Samoa,1986,0.005785,0.116260,0.256774,0.610622,0.298980,0.191413,0.120229
7,ASM,American Samoa,1987,0.005546,0.111139,0.252195,0.621116,0.299544,0.196242,0.125330
8,ASM,American Samoa,1988,0.005318,0.106418,0.247702,0.631072,0.299851,0.200815,0.130406
9,ASM,American Samoa,1989,0.005104,0.102126,0.243342,0.640414,0.299893,0.205063,0.135458


## 3. Quality assurance

In [3]:
print(bmi_pacific.info())

print()

print("Countries:")
print(sorted(bmi_pacific["Country"].unique()))

print()

print("Years:")
print(bmi_pacific["Year"].min(), "to", bmi_pacific["Year"].max())

print()

print("Missing values by column:")
print(bmi_pacific.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ISO3                720 non-null    object 
 1   Country             720 non-null    object 
 2   Year                720 non-null    int64  
 3   Underweight_Pct     720 non-null    float64
 4   HealthyWeight_Pct   720 non-null    float64
 5   Overweight_Pct      720 non-null    float64
 6   Obesity_Pct         720 non-null    float64
 7   Class1_Obesity_Pct  720 non-null    float64
 8   Class2_Obesity_Pct  720 non-null    float64
 9   Morbid_Obesity_Pct  720 non-null    float64
dtypes: float64(7), int64(1), object(2)
memory usage: 56.4+ KB
None

Countries:
['American Samoa', 'Cook Islands', 'Federated States of Micronesia', 'Fiji', 'French Polynesia', 'Kiribati', 'Marshall Islands', 'Nauru', 'Niue', 'Palau', 'Samoa', 'Solomon Islands', 'Tokelau', 'Tonga', 'Tuvalu', 'Vanuatu']

Ye

## 4. Coverage assessment

In [4]:
coverage = (
    bmi_pacific
    .groupby(["ISO3", "Country"])
    .agg(
        First_Year=("Year", "min"),
        Last_Year=("Year", "max"),
        Years=("Year", "count")
    )
    .reset_index()
)

coverage

,ISO3,Country,First_Year,Last_Year,Years
0,ASM,American Samoa,1980,2024,45
1,COK,Cook Islands,1980,2024,45
2,FJI,Fiji,1980,2024,45
3,FSM,Federated States of Micronesia,1980,2024,45
4,KIR,Kiribati,1980,2024,45
5,MHL,Marshall Islands,1980,2024,45
6,NIU,Niue,1980,2024,45
7,NRU,Nauru,1980,2024,45
8,PLW,Palau,1980,2024,45
9,PYF,French Polynesia,1980,2024,45


## 5. Export cleaned dataset

In [5]:
bmi_pacific.to_csv(
    "processed_data/adult_bmi_distribution.csv",
    index=False
)

coverage.to_csv(
    "processed_data/adult_bmi_distribution_coverage.csv",
    index=False
)

print("Notebook 4 complete.")
print("Files saved:")
print("- processed_data/adult_bmi_distribution.csv")
print("- processed_data/adult_bmi_distribution_coverage.csv")

Notebook 4 complete.
Files saved:
- processed_data/adult_bmi_distribution.csv
- processed_data/adult_bmi_distribution_coverage.csv


# Notebook Summary

This notebook successfully:

- Downloaded NCD-RisC age-standardized adult BMI distribution data.
- Filtered to the 16 Pacific Island jurisdictions.
- Averaged male and female prevalence estimates to create a single country-year dataset.
- Produced seven BMI distribution indicators:
  - Underweight
  - Healthy weight
  - Overweight
  - Obesity
  - Class I obesity
  - Class II obesity
  - Morbid obesity
- Verified complete coverage from 1980–2024 with no missing values.
- Exported the cleaned dataset for downstream analysis.